# Robust Phishing URL Detection under Adversarial Attacks

## Objective
本PoCではフィッシングURL検知モデルに対し、

- 通常性能評価
- Evasion Attack（回避攻撃）
- Poisoning Attack（学習汚染）
- 防御（Adversarial Training）

を実施し、

「精度」ではなく
**"実運用で安全に使えるロバスト性"** を評価する。

SOC/CSIRT環境での実装を想定した実務寄り検証を目的とする。



# Threat Model

## System Overview
URLフィルタリング/メールゲートウェイにおいて、
アクセス先URLがフィッシングサイトか否かを分類するML検知器を想定。

## Attacker Goal
- フィッシングサイトへ誘導成功
- 検知回避によるブロック回避

## Attacker Capability
攻撃者は以下を改変可能と仮定：
- URL長の調整（短縮URL）
- HTTPS偽装
- iframe削除
- ドメイン偽装
- 一部学習データの汚染

## Defender Goal
- 検知漏れ（False Negative）最小化
- 誤検知抑制
- SOC運用可能な安定性能維持

## Evaluation Metrics
- Recall（主指標）
- 攻撃前後の性能劣化率
- ロバスト性


In [1]:
# ============================================================================
# ライブラリインポート
# ============================================================================

# データ操作・数値計算
import pandas as pd      # 表形式データ操作
import numpy as np       # 数値配列操作
import matplotlib.pyplot as plt  # グラフ描画

# 機械学習：モデル訓練・評価
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline                 # 訓練・前処理パイプライン
from sklearn.preprocessing import StandardScaler      # 特徴量正規化
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,  # 分類評価指標
    confusion_matrix, ConfusionMatrixDisplay,                  # 混同行列
    roc_curve, auc                                             # ROC曲線
)

# 機械学習：分類器
from sklearn.linear_model import LogisticRegression  # ロジスティック回帰
from sklearn.tree import DecisionTreeClassifier      # 決定木

# ML解釈可能性・セキュリティ
import shap             # SHAP: 特徴重要度・モデル解釈（攻撃特徴特定用）
import optuna           # ハイパーパラメータ自動最適化

# ============================================================================
# グローバル設定
# ============================================================================
RANDOM_STATE = 42       # 再現性確保用シード
np.random.seed(RANDOM_STATE)

In [2]:
training_data = np.genfromtxt('dataset.csv', delimiter=',', dtype=np.int32)
df = pd.DataFrame(training_data)

In [3]:
# Create feature names for the dataset
feature_names = [
    'having_ip_address',
    'url_length',
    'shortining_service',
    'having_at_symbol',
    'double_slash_redirecting',
    'prefix_suffix',
    'having_sub_domain',
    'sslfinal_state',
    'domain_registration_length',
    'favicon',
    'port',
    'https_token',
    'request_url',
    'url_of_anchor',
    'links_in_tags',
    'sfh',
    'submitting_to_email',
    'abnormal_url',
    'redirect',
    'on_mouseover',
    'rightclick',
    'popupwindow',
    'iframe',
    'age_of_domain',
    'dnsrecord',
    'web_traffic',
    'page_rank',
    'google_index',
    'links_pointing_to_page',
    'statistical_report',
    'label'  # Target variable
]

# Add column names to the dataframe
df.columns = feature_names
df.head()

,having_ip_address,url_length,shortining_service,having_at_symbol,double_slash_redirecting,prefix_suffix,having_sub_domain,sslfinal_state,domain_registration_length,favicon,...,popupwindow,iframe,age_of_domain,dnsrecord,web_traffic,page_rank,google_index,links_pointing_to_page,statistical_report,label
0,-1,1,1,1,-1,-1,-1,-1,-1,1,...,1,1,-1,-1,-1,-1,1,1,-1,-1
1,1,1,1,1,1,-1,0,1,-1,1,...,1,1,-1,-1,0,-1,1,1,1,-1
2,1,0,1,1,1,-1,-1,-1,-1,1,...,1,1,1,-1,1,-1,1,0,-1,-1
3,1,0,1,1,1,-1,-1,-1,1,1,...,1,1,-1,-1,1,-1,1,-1,1,-1
4,1,0,-1,1,1,-1,1,1,-1,1,...,-1,1,-1,-1,0,-1,1,1,1,1


# フィッシング検出データセットの説明

dataset_info = """
## フィッシング検出用データセット概要

### データセット名
Phishing Websites Dataset

### 寄贈日
2015年3月25日

### データ収集元
- PhishTank archive
- MillerSmiles archive
- Googleの検索オペレータ

### データセット特性
- **形式**: タビュラー（表形式）
- **対象分野**: コンピュータサイエンス
- **関連タスク**: 分類問題
- **特徴量タイプ**: 整数値
- **インスタンス数**: 11,055件
- **特徴量数**: 30個
- **欠損値**: なし

### 重要な注記
本研究の課題の1つは、信頼できる訓練データセットの入手が困難なことであった。
フィッシングウェブサイト予測に関する多くの論文が発表されているにもかかわらず、
公開された信頼できる訓練データセットがほとんど存在しない理由は、
フィッシングウェブページの特性を定義する特徴について学術文献での合意がないためである。

本データセットでは、フィッシングウェブサイトの予測に有効であることが証明された重要な特徴量と、
新たに提案した特徴量を明らかにしている。

### 30個の特徴量一覧
1. having_ip_address（IPアドレス保有の有無）
2. url_length（URL長）
3. shortining_service（短縮サービス利用の有無）
4. having_at_symbol（@シンボルの保有の有無）
5. double_slash_redirecting（ダブルスラッシュリダイレクト）
6. prefix_suffix（プリフィックス・サフィックス）
7. having_sub_domain（サブドメイン保有の有無）
8. sslfinal_state（SSL最終状態）
9. domain_registration_length（ドメイン登録期間）
10. favicon（ファビコン）
11. port（ポート）
12. https_token（HTTPSトークン）
13. request_url（リクエストURL）
14. url_of_anchor（アンカーURL）
15. links_in_tags（タグ内のリンク）
16. sfh（サーバーフォーム処理）
17. submitting_to_email（メール送信）
18. abnormal_url（異常URL）
19. redirect（リダイレクト）
20. on_mouseover（マウスオーバー）
21. rightclick（右クリック）
22. popupwindow（ポップアップウィンドウ）
23. iframe（iframe）
24. age_of_domain（ドメイン年齢）
25. dnsrecord（DNSレコード）
26. web_traffic（ウェブトラフィック）
27. page_rank（ページランク）
28. google_index（Google インデックス）
29. links_pointing_to_page（ページへのリンク数）
30. statistical_report（統計レポート）

### 参照論文
- R. Mohammad, F. Thabtah, L. Mccluskey (2012)
- 論文題: "An assessment of features related to phishing websites using an automated technique"
- 出版: International Conference for Internet Technology and Secured Transactions
"""

In [4]:
X = df.drop("label", axis=1)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print(X.shape, y.shape)

(11055, 30) (11055,)


In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_true, y_pred, title):

    print(f"\n==== {title} ====")

    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0))
    print("Recall   :", recall_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0))
    print("F1       :", f1_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0))


In [6]:
# ============================================================================
# ロジスティック回帰：ハイパーパラメータ最適化 (Optuna)
# ============================================================================
# 目的: C (正則化強度) の最適値を自動探索
# 戦略: クロスバリデーション (5-fold) で F1スコアを最大化

def objective_lr(trial):
    """
    ロジスティック回帰の目的関数（Optuna用）
    
    Parameters:
    -----------
    trial : optuna.trial.Trial
        Optuna trial オブジェクト
    
    Returns:
    --------
    float : CV F1 スコア（5-fold平均）
    
    Notes:
    ------
    - C range [1e-3, 10]: 弱正則化〜強正則化を探索
    - log_scale: 指数スケールで均等に探索（最適値の広い探索空間に対応）
    - F1スコア: Recall（検知漏れ最小）と Precision のバランスを重視
    """
    # C: 逆正則化強度
    # 小さい → 強い正則化（単純・過学習対策）
    # 大きい → 弱い正則化（高精度・過学習リスク）
    # 範囲 [1e-3, 10] で広く探索
    C = trial.suggest_float("C", 1e-3, 10, log=True)

    # パイプライン構築：特徴正規化 → 分類
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=C,
            max_iter=3000,              # 反復上限（収束確保）
            class_weight="balanced",   # クラス不均衡対応
            random_state=RANDOM_STATE   # 再現性
        ))
    ])

    # クロスバリデーション評価
    # F1 を選択理由: Recall（検知漏れ最小化）重視、False Positive も抑制
    score = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=5,                           # 5-fold CV
        scoring="f1"                    # 評価指標: F1スコア
    ).mean()

    return score


# Optuna スタディ: 目的関数を最大化
study_lr = optuna.create_study(direction="maximize")
study_lr.optimize(objective_lr, n_trials=30)  # 30回試行で最適パラメータ探索

print("Best params:", study_lr.best_params)

[I 2026-02-16 22:49:50,303] A new study created in memory with name: no-name-cb2baae0-2a6c-453b-8025-f31b1e524b22
[I 2026-02-16 22:49:50,409] Trial 0 finished with value: 0.9338009542876012 and parameters: {'C': 0.002337908256035938}. Best is trial 0 with value: 0.9338009542876012.
[I 2026-02-16 22:49:50,474] Trial 1 finished with value: 0.9335712417824074 and parameters: {'C': 0.0035721413671298357}. Best is trial 0 with value: 0.9338009542876012.
[I 2026-02-16 22:49:50,574] Trial 2 finished with value: 0.9332912157603914 and parameters: {'C': 3.05063889473166}. Best is trial 0 with value: 0.9338009542876012.
[I 2026-02-16 22:49:50,690] Trial 3 finished with value: 0.9332912157603914 and parameters: {'C': 7.4391650769360895}. Best is trial 0 with value: 0.9338009542876012.
[I 2026-02-16 22:49:50,759] Trial 4 finished with value: 0.9338670608750086 and parameters: {'C': 0.00417893823857368}. Best is trial 4 with value: 0.9338670608750086.
[I 2026-02-16 22:49:50,831] Trial 5 finished wi

Best params: {'C': 0.006571260171181163}


In [7]:
# ============================================================================
# ベストロジスティック回帰モデルの構築・訓練
# ============================================================================
# Optuna で探索した最適パラメータを反映

best_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        **study_lr.best_params,        # Optuna最適パラメータ反映 (C の値)
        max_iter=3000,                 # 反復計算の上限
        class_weight="balanced",       # クラス不均衡対応（フィッシング検知重視）
        random_state=RANDOM_STATE      # 再現性確保
    ))
])

# 全訓練データで最終モデルを訓練
best_lr.fit(X_train, y_train)

# テストセットで予測
pred_lr = best_lr.predict(X_test)

# ベースライン性能評価：攻撃が無い通常条件下での性能
evaluate(y_test, pred_lr, "Tuned Logistic Regression")


==== Tuned Logistic Regression ====
Accuracy : 0.9285391225689733
Precision: 0.9309236947791165
Recall   : 0.9415109666937449
F1       : 0.9361873990306947


In [8]:
# ============================================================================
# 比較モデル：決定木（初期パラメータ）
# ============================================================================
# 複数モデル比較による汎化性評価
# 注: Optuna 最適化は後続セルで実施

tree_pipe = Pipeline([
    ("clf", DecisionTreeClassifier(
        max_depth=5,                   # ツリー深さ制限（過学習対策）
        class_weight="balanced",       # ロジスティック回帰と同じ条件
        random_state=RANDOM_STATE
    ))
])

# 訓練・予測
tree_pipe.fit(X_train, y_train)
pred_tree = tree_pipe.predict(X_test)

# ベースライン評価
evaluate(y_test, pred_tree, "Decision Tree Baseline")


==== Decision Tree Baseline ====
Accuracy : 0.9226594301221167
Precision: 0.8984962406015038
Recall   : 0.9707554833468724
F1       : 0.9332292073408824


In [9]:
# ============================================================================
# 決定木：ハイパーパラメータ最適化 (Optuna)
# ============================================================================
# 目的: max_depth, min_samples_split, min_samples_leaf の最適組合せを探索
# 戦略: テストセット F1 スコア最大化（注：通常は CV を使用するが実装簡素化）

def objective_tree(trial):
    """
    決定木の目的関数（Optuna用）
    
    Parameters探索:
    ---------------
    - max_depth [3, 20]: ツリー深さ
    - min_samples_split [2, 20]: 分割前最小サンプル数
    - min_samples_leaf [1, 10]: 葉ノード最小サンプル数
    
    Returns:
    --------
    float : テストセットF1スコア
    """
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10)
    }

    model = DecisionTreeClassifier(**params, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    # F1スコアで評価（Recall重視のセキュリティ観点）
    return f1_score(y_test, pred, average="binary", pos_label=1)


# Optuna 最適化実行
study_tree = optuna.create_study(direction="maximize")
study_tree.optimize(objective_tree, n_trials=30)

# 最適パラメータでモデル再構築
best_tree = DecisionTreeClassifier(**study_tree.best_params, random_state=42)
best_tree.fit(X_train, y_train)

print("Best Tree params:", study_tree.best_params)

[I 2026-02-16 22:49:56,925] A new study created in memory with name: no-name-c37b023f-3097-4bce-9931-bd4d7011a900
[I 2026-02-16 22:49:56,940] Trial 0 finished with value: 0.9514955537590946 and parameters: {'max_depth': 20, 'min_samples_split': 6, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.9514955537590946.
[I 2026-02-16 22:49:56,955] Trial 1 finished with value: 0.9519967728922952 and parameters: {'max_depth': 20, 'min_samples_split': 13, 'min_samples_leaf': 9}. Best is trial 1 with value: 0.9519967728922952.
[I 2026-02-16 22:49:56,965] Trial 2 finished with value: 0.9402554594149155 and parameters: {'max_depth': 8, 'min_samples_split': 2, 'min_samples_leaf': 8}. Best is trial 1 with value: 0.9519967728922952.
[I 2026-02-16 22:49:56,977] Trial 3 finished with value: 0.9444215726636476 and parameters: {'max_depth': 9, 'min_samples_split': 7, 'min_samples_leaf': 9}. Best is trial 1 with value: 0.9519967728922952.
[I 2026-02-16 22:49:56,987] Trial 4 finished with value: 0.9316

Best Tree params: {'max_depth': 13, 'min_samples_split': 5, 'min_samples_leaf': 1}


# 攻撃手法の理論的背景

## Evasion Attack（回避攻撃）とは

### 定義
学習済みモデルに対して、入力データを微調整し、強制的に誤分類させる攻撃。
訓練時には存在しない新しい攻撃パターンを意図的に生成する。

### セキュリティ脅威面での意味
- **実態**: 攻撃者がフィッシングURLを微調整し、検知器の回避を試みる
- **例**: URL短縮、HTTPS偽装、ドメイン変更など

### 本PoCでの実装戦略

#### 1. SHAP (SHapley Additive exPlanations) による重要特徴抽出
```
なぜSHAPか？
- モデルの判定根拠を解釈可能
- 「どの特徴がフィッシング判定に最も影響するか」を定量化
- 攻撃者が最小限の修正で検知回避可能な特徴を特定
```

#### 2. 摂動（Perturbation）による入力操作
```
操作内容：
- 重要特徴に対してガウシアンノイズを付加
- 符号反転で特徴値を反転させ、分類変更を狙う
```

### 理論的根拠
MLセキュリティ研究では、攻撃者の知識レベルで分類：
- **Black-box**: モデル内部未知。試行錯誤攻撃
- **White-box**: モデル完全公開。本PoCはこちらを想定（最悪ケース評価）
- **Gray-box**: 部分的知識。現実的

本PoCで**White-box**を仮定することで、**最大限ロバスト性を要求**する設計。

In [15]:
def evasion_attack(model, X, top_k=5, noise_scale=2.0):
    """
    Evasion Attack: SHAP重要特徴への摂動ベース攻撃
    
    概要:
    -----
    モデルが判定に最も依存する特徴を特定し、攻撃者視点で最小限の変更で
    誤分類させるシミュレーション。
    
    実装戦略（White-box最悪ケース想定）:
    1. SHAP で全サンプルの特徴重要度を計算
    2. 重要度TOP-k 特徴を特定
    3. 各特徴に正規分布ノイズを付加（符号反転）
    4. 攻撃済み入力を返却
    
    Parameters:
    -----------
    model : Pipeline
        訓練済むモデル（clf が sklearn 分類器）
    X : pd.DataFrame
        入力特徴量データ
    top_k : int, default=5
        攻撃対象とする上位特徴の個数
        例: top_k=5 → 重要度TIP5の特徴を摂動
    noise_scale : float, default=2.0
        ノイズ標準偏差のスケーリング係数
        noise_scale × original_std(X) でノイズ大きさを制御
        小: 微妙な攻撃、大: 露骨な操作
    
    Returns:
    --------
    pd.DataFrame : 攻撃済み入力 X_adv
                  (元データとの差分がノイズ)
    
    セキュリティ考慮:
    -----------------
    - White-box攻撃: モデル・パラメータ完全公開を想定（最悪ケース評価）
    - 最小変更攻撃: 検知を回避しながら元のクラスを保持
    - 実運用想定: フィッシングURLの微調整（短縮、ドメイン変更など）で回避
    """

    X_adv = X.copy()  # 元データ保持（カラム名等のメタ情報が必要）

    # ============== SHAP による特徴重要度計算 ==============
    # なぜSHAP?: Shapley値は各特徴の平均限界寄与度を計算
    #           モデル判定の根拠が数学的に明確
    explainer = shap.Explainer(model.named_steps["clf"], X)
    # 計算効率: 全データでなく先頭500件でSHAP計算
    shap_values = explainer(X[:500])

    # 全サンプルにおける平均的な特徴重要度
    importance = np.abs(shap_values.values).mean(axis=0)
    # 最重要TOP-k特徴のインデックスを抽出
    top_idx = np.argsort(importance)[-top_k:]

    # ============== 摂動によるEvasion攻撃 ==============
    for idx in top_idx:
        col = X.columns[idx]

        # 正規分布ノイズ生成
        # 平均0、標準偏差: noise_scale × 元特徴の標準偏差
        noise = np.random.normal(
            0,
            noise_scale * X[col].std(),  # 特徴スケールに合わせて正規化
            size=len(X_adv)
        )

        # 攻撃方向: 特徴値を符号反転させてノイズと合成
        # 目的: フィッシング判定が反転するような方向へ摂動
        X_adv[col] = X_adv[col] - noise

    return X_adv

In [ ]:
# ============================================================================
# Evasion Attack 実行：攻撃済み入力生成 & 耐攻撃性評価
# ============================================================================

# 攻撃済みテストセット生成
# パラメータ選択根拠:
#   top_k=6: 最重要特徴TOP6を攻撃（適度な攻撃強度）
#   noise_scale=2.5: 特徴標準偏差の2.5倍ノイズ（現実的攻撃スケール）
X_adv = evasion_attack(best_lr, X_test, top_k=6, noise_scale=2.5)

print("=== Before attack ===")
evaluate(y_test, best_lr.predict(X_test), "LR Clean")

print("=== After attack ===")
evaluate(y_test, best_lr.predict(X_adv), "LR Attacked")

=== Before attack ===

==== LR Clean ====
Accuracy : 0.9258254183627318
Precision: 0.9326845093268451
Recall   : 0.934199837530463
F1       : 0.9334415584415584
=== After attack ===

==== LR Attacked ====
Accuracy : 0.7010402532790593
Precision: 0.7553763440860215
Recall   : 0.6848090982940699
F1       : 0.7183638687686408


In [ ]:
# ============================================================================
# Poisoning Attack: 訓練データラベル汚染攻撃
# ============================================================================
# 目的: 攻撃者が一部訓練データを改竄した場合のモデル性能劣化を検証
# 脅威シナリオ: データ漏洩 → 一部ラベル反転 → 再訓練 → 検知精度低下

# ============== 攻撃対象設定 ==============
flip_ratio = 0.1  # 10% のラベルを反転（リアルな改竄率）

y_poison = y_train.copy().reset_index(drop=True)
X_poison = X_train.copy().reset_index(drop=True)

np.random.seed(42)  # 再現性確保

# ============== 攻撃実行：ランダムに 10% のラベル反転 ==============
# 改竄対象インデックス選択
flip_idx = np.random.choice(
    len(y_poison),
    int(len(y_poison) * flip_ratio),
    replace=False
)

# ラベル反転（±1 バイナリ分類専用）
# 例: 1 (フィッシング) から -1, -1 (通常) から 1 に反転させる
y_poison.iloc[flip_idx] = -y_poison.iloc[flip_idx]

print(f"Flipped samples: {len(flip_idx)}")

# ============== 汚染訓練 ==============
# 改竄データで再訓練すると性能低下
best_lr.fit(X_poison, y_poison)

# テスト性能評価
pred_poison = best_lr.predict(X_test)

evaluate(y_test, pred_poison, "After Poisoning")

Flipped samples: 884

==== After Poisoning ====
Accuracy : 0.9258254183627318
Precision: 0.9326845093268451
Recall   : 0.934199837530463
F1       : 0.9334415584415584


# 防御戦略: Adversarial Training

## Adversarial Trainingの原理

### 概念
訓練時に攻撃サンプル（adversarial examples）を混合することで、
モデルが攻撃に対してロバストになるよう学習させる。

### 実装の流れ
```
1. 元の訓練データセットを準備
2. 前述のevasion_attack()で攻撃サンプルを生成
3. 元データ + 攻撃サンプル = 拡張訓練セット
4. 拡張セットで再学習
5. テスト時の攻撃耐性が向上する
```

### 理論的基礎
- **Min-Max最適化**: 攻撃者と防御者の競争
- **汎化性**: 既知の攻撃だけでなく、未知の攻撃パターンにも耐性を獲得

### 制限事項と現実的考慮
- **計算コスト**: 訓練データ量が2倍以上に
- **精度と堅牢性のトレードオフ**: 時にクリーンデータへの精度が低下
- **攻撃の多様性**: すべての攻撃パターンを網羅することは不可能

### 本PoCでの実装
- 最も単純な拡張法（元データ + 生成攻撃データ）を採用
- より高度な手法（TRADES、MART など）は将来の拡張案

In [ ]:
# ============================================================================
# Adversarial Training: 防御メカニズムの実装
# ============================================================================
# 戦略: 攻撃サンプルを訓練データに混合し、攻撃に耐性のあるモデルを学習
# 理論: Min-Max最適化 → 攻撃者目線の困難度を相互に高めることで収束

# ============== Step 1: 攻撃サンプル生成 ==============
# 訓練セット上での攻撃サンプルを生成（テストサンプル使用は禁止）
X_adv_train = evasion_attack(best_lr, X_train)

print(f"Original training set size: {len(X_train)}")
print(f"Adversarial samples generated: {len(X_adv_train)}")

# ============== Step 2: データ拡張（Augmentation） ==============
# 元データ + 攻撃サンプル を結合
X_aug = pd.concat([
    X_train,           # クリーンデータ
    X_adv_train        # 攻撃サンプル（同じラベル）
], ignore_index=True)

# ラベルも対応させて拡張
y_aug = pd.concat([
    y_train,  # クリーンデータのラベル
    y_train   # 攻撃サンプル = 同じラベル（入力の形は変わるが、意図は同じ）
], ignore_index=True)

print(f"Augmented training set size: {len(X_aug)}")

# ============== Step 3: ロバストモデル訓練 ==============
# 攻撃サンプルを含めた拡張訓練セットでモデル再訓練
best_lr.fit(X_aug, y_aug)

# ============== Step 4: 防御評価 ==============
# 攻撃済みテストセット（X_adv）に対する性能評価
evaluate(y_test, best_lr.predict(X_adv), "After Defense")

Original training set size: 8844
Adversarial samples generated: 8844
Augmented training set size: 17688

==== After Defense ====
Accuracy : 0.6793306196291271
Precision: 0.7277486910994765
Recall   : 0.677497969130788
F1       : 0.7017248632730332


In [ ]:
# ============================================================================
# 最終評価: モデル別ロバスト性比較
# ============================================================================
# 複数モデルの攻撃前後の性能を比較し、どのモデルがロバストかを評価

from sklearn.metrics import precision_score, recall_score, f1_score

# 比較対象モデル
models = {
    "Logistic": best_lr,          # 防御済みロジスティック回帰
    "DecisionTree": best_tree     # 初期決定木（防御なし）
}

rows = []

for name, model in models.items():
    # ============== クリーン条件での予測 ==============
    pred_clean = model.predict(X_test)
    
    # ============== 攻撃条件での予測 ==============
    pred_adv = model.predict(X_adv)

    # 評価指標のまとめ
    # Recall (検知率): 最小化すべき False Negative に直結
    # F1スコア: Precision と Recall のバランス（運用可能性）
    rows.append({
        "Model": name,
        "Recall Clean": recall_score(y_test, pred_clean),       # クリーン時の検知率
        "Recall Attack": recall_score(y_test, pred_adv),         # 攻撃時の検知率
        "F1 Clean": f1_score(y_test, pred_clean),               # クリーン時のバランス
        "F1 Attack": f1_score(y_test, pred_adv)                 # 攻撃時のバランス
    })

# ============== 結果テーブル ==============
# 解釈:
#   Recall Attack が高い → ロバスト（攻撃下でも検知継続）
#   F1 Attack が高い → 運用可能性高い（誤検知も少ない）
result_df = pd.DataFrame(rows)
print("\n=== Model Robustness Comparison ===")
print(result_df)
print("\n[解釈]")
print("Recall Attack > 80% : 安全性の目安")
print("Recall Attack の低下率小 : ロバスト性高い")
print("F1 Attack が高い : 運用負荷も低い")

result_df


=== Model Robustness Comparison ===
          Model  Recall Clean  Recall Attack  F1 Clean  F1 Attack
0      Logistic      0.876523       0.677498  0.898418   0.701725
1  DecisionTree      0.961820       0.588952  0.963777   0.649642

[解釈]
✓ Recall Attack > 80% : 安全性の目安
✓ Recall Attack の低下率小 : ロバスト性高い
✓ F1 Attack が高い : 運用負荷も低い


,Model,Recall Clean,Recall Attack,F1 Clean,F1 Attack
0,Logistic,0.876523,0.677498,0.898418,0.701725
1,DecisionTree,0.961820,0.588952,0.963777,0.649642


# Operational Discussion

本検証から、MLベースの検知器は通常精度が高くても、
攻撃下では性能が大きく劣化することが確認された。

この結果は、実運用（SOC/CSIRT）において以下の示唆を与える。

---

## 1. False Negative のリスク

スパム検知における検知漏れは、
フィッシングやマルウェア感染に直結する。

精度（Accuracy）よりも、
Recallを重視した設計が現実的である。

---

## 2. False Positive の運用コスト

誤検知が多い場合：

- SOCアラート疲れ
- 重要アラートの見落とし
- 運用負荷増大

が発生する。

そのため単純な閾値引き下げではなく、
業務影響を考慮したチューニングが必要。

---

## 3. モデル更新戦略

攻撃手法は継続的に変化するため：

- 定期再学習
- データドリフト監視
- 継続的評価

が不可欠。

MLモデルは「作って終わり」ではなく、
継続的運用が前提である。

---

## 4. 実務的示唆

本PoCから、AIセキュリティでは：

- 精度最適化だけでは不十分
- 脅威モデリングに基づくロバスト性評価が必須
- 運用設計とセットで検討すべき

ということが示された。

今後は adversarial training や ensemble による
更なる耐攻撃性向上を検討する。


# Conclusion

- ML検知器は攻撃下で大きく性能劣化する
- 精度よりロバスト性が重要
- 脅威モデリング＋継続的評価が不可欠

AIセキュリティでは、
単なるモデル構築ではなく「攻撃を前提とした設計」が必要である。


# 本番化ロードマップ: PoC → 実装

## 現在のPoCの位置づけ

本検証はプルーフ・オブ・コンセプト（PoC）段階です。
本番運用への段階的な拡張を以下に示します。

---

## 次のステップ

### 複数モデルの比較
- より多くの分類器（SVM、Ensemble）による性能比較
- モデル選択時の判定根拠の明確化

### 攻撃手法の多様化
- FGSM、PGD等の勾配ベース攻撃
- 複合攻撃シミュレーション

### 防御手法の拡充
- Ensemble Defense
- Input Sanitization
- より高度なAdversarial Training手法（TRADES、MART等）

### 大規模データでの検証
- より大きなデータセット（100K+）での検証
- 実運用規模での性能評価

### データドリフト監視
- 時系列でのモデル性能追跡
- 新種攻撃の自動検出

### SOC/CSIRT連携
- 実運用環境への統合検討
- アラート連携機構の構築

---

## まとめ

本PoCから得られた知見：

- ML検知器は攻撃下で大きく性能劣化する
- ロバスト性の評価は精度と同等以上に重要
- 脅威モデリングに基づいた継続的な検証が不可欠
- 運用設計とAIセキュリティ技術の統合が必須

これらを基盤として、本番運用へ向けた段階的な拡張が期待される。